In [ ]:
%pip install -U ipywidgets

In [ ]:
%pip install git+https://github.com/py-why/causal-learn.git

  Cloning https://github.com/py-why/causal-learn.git to c:\users\hp\appdata\local\temp\pip-req-build-ghfbp0wi
  Resolved https://github.com/py-why/causal-learn.git to commit 9689c1bdc468847729eacf0921b76f598161ae16
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'


  Running command git clone --filter=blob:none --quiet https://github.com/py-why/causal-learn.git 'C:\Users\HP\AppData\Local\Temp\pip-req-build-ghfbp0wi'


In [1]:
import numpy as np
import pandas as pd
import scipy
import os
import time

In [3]:
# Load your data
# required_data = np.load("requiredDataFullUpdated.npy")
# print(required_data.shape)
# # Define variable names
givenParams = ['WOW', 'LATP', 'LONP', 'BAL1', 'ROLL', 'PTCH', 'IVV', 'GS', 'CAS', 'VRTG', 'LATG', 'FLAP', 'PLA_2', 'PLA_3', 'PLA_4', 'TRK', 'TH', 'WS', 'WD', 'TAT', 'SAT', 'LOC', 'GLS']
var_names = [f"{param}_mean" for param in givenParams] + [f"{param}_std" for param in givenParams] + ["TouchdownDistance"]
# # Create a dataframe from this numpy array
# df = pd.DataFrame(data = required_data, columns= var_names)
# df.head(10)

df = pd.read_csv('FullFilteredRequiredData.csv')
print(df.shape)
df.head()

(45902, 47)


,WOW_mean,LATP_mean,LONP_mean,BAL1_mean,ROLL_mean,PTCH_mean,IVV_mean,GS_mean,CAS_mean,VRTG_mean,...,PLA_4_std,TRK_std,TH_std,WS_std,WD_std,TAT_std,SAT_std,LOC_std,GLS_std,TouchdownDistance
0,0.952381,46.907830,-96.815466,926.583333,0.308523,0.350309,-429.431548,110.041667,116.068452,1.009552,...,4.292836,0.237390,0.374503,0.861338,7.157449,1.022008,1.213352,0.003225,0.169581,1639.915003
1,0.952381,35.058233,-89.953914,320.095238,-1.124300,-0.782557,-501.000000,119.778274,120.196429,1.001922,...,8.545955,0.578280,2.304534,4.595623,63.336208,0.976916,1.244717,0.012306,0.070444,1405.330113
2,0.952381,42.227989,-83.365766,704.011905,0.409032,0.227174,-422.184524,116.321429,116.225446,1.000219,...,9.463613,0.500724,1.202374,2.575892,7.046211,1.062396,1.218829,0.001691,0.234278,1320.815633
3,0.952381,38.042018,-84.596677,1043.773810,-0.065328,0.299499,-360.699405,113.281250,113.395833,0.790925,...,8.860344,0.377983,1.607919,2.038931,149.417796,1.128381,1.235918,0.015060,0.295813,1514.608794
4,0.952381,41.538736,-93.651606,977.035714,-0.348021,-0.642289,-349.883929,117.413690,119.049107,1.002454,...,8.689983,0.219237,1.207223,3.624479,98.996618,0.929944,1.218829,0.026006,0.151445,2067.536266


### Using the __LINGAM__ Algorithm

In [3]:
from causalai.models.tabular.lingam import LINGAM

from causalai.data.data_generator import DataGenerator
# also importing data object, data transform object, and prior knowledge object, and the graph plotting function
from causalai.data.tabular import TabularData
from causalai.data.transforms.tabular import StandardizeTransform
from causalai.models.common.prior_knowledge import PriorKnowledge
from causalai.misc.misc import plot_graph, get_precision_recall, get_precision_recall_skeleton, make_symmetric

In [4]:
# 1.
StandardizeTransform_ = StandardizeTransform()
StandardizeTransform_.fit(df.to_numpy())

data_trans = StandardizeTransform_.transform(df.to_numpy())

# 2.
data_obj = TabularData(data_trans, var_names=var_names)

In [ ]:
prior_knowledge = None

lingam = LINGAM(
        data=data_obj,
        prior_knowledge=prior_knowledge,
        )
tic = time.time()
result = lingam.run(pvalue_thres=0.05)
toc = time.time()
print(f'Time taken: {toc-tic:.2f}s\n')


print(f' The output result has keys: {result.keys()}')
print(f' The output result["WOW_mean"] has keys: {result["WOW_mean"].keys()}')

In [ ]:
print(f'Predicted parents:')
result['TouchdownDistance']['parents']

Predicted parents:


['IVV_mean',
 'GS_std',
 'WS_mean',
 'PTCH_std',
 'WD_mean',
 'CAS_mean',
 'WOW_std',
 'WS_std',
 'WOW_mean',
 'TAT_std',
 'SAT_std',
 'TH_std',
 'VRTG_std',
 'FLAP_mean',
 'TRK_std',
 'VRTG_mean',
 'WD_std',
 'PTCH_mean',
 'ROLL_mean']

__Causal Strengths__

In [ ]:
causalStrengthsDictLINGAM = {k:result['TouchdownDistance']['value_dict'][k] for k in result['TouchdownDistance']['parents']}
causalStrengthsDictLINGAM

{'IVV_mean': np.float64(0.3211581691323262),
 'GS_std': np.float64(0.2779041298681629),
 'WS_mean': np.float64(-0.20108171538526104),
 'PTCH_std': np.float64(-0.10517237598253339),
 'WD_mean': np.float64(0.10424809346377813),
 'CAS_mean': np.float64(0.09361652572651555),
 'WOW_std': np.float64(0.09152030645654459),
 'WS_std': np.float64(0.08858735404378891),
 'WOW_mean': np.float64(0.07038263245857962),
 'TAT_std': np.float64(0.06414859678372849),
 'SAT_std': np.float64(-0.05702405522729671),
 'TH_std': np.float64(0.04956813639417614),
 'VRTG_std': np.float64(-0.037226501508413334),
 'FLAP_mean': np.float64(-0.035909836120497474),
 'TRK_std': np.float64(0.03333559841745487),
 'VRTG_mean': np.float64(-0.03199037970377651),
 'WD_std': np.float64(0.019952000406363777),
 'PTCH_mean': np.float64(0.017416942412486425),
 'ROLL_mean': np.float64(0.010664611750704302)}

In [ ]:
# Sorting the dictionary by absolute value in descending order
sorted_dict = dict(sorted(causalStrengthsDictLINGAM.items(), key=lambda item: abs(item[1]), reverse=True))

# Printing the sorted dictionary
sorted_dict

{'IVV_mean': np.float64(0.3211581691323262),
 'GS_std': np.float64(0.2779041298681629),
 'WS_mean': np.float64(-0.20108171538526104),
 'PTCH_std': np.float64(-0.10517237598253339),
 'WD_mean': np.float64(0.10424809346377813),
 'CAS_mean': np.float64(0.09361652572651555),
 'WOW_std': np.float64(0.09152030645654459),
 'WS_std': np.float64(0.08858735404378891),
 'WOW_mean': np.float64(0.07038263245857962),
 'TAT_std': np.float64(0.06414859678372849),
 'SAT_std': np.float64(-0.05702405522729671),
 'TH_std': np.float64(0.04956813639417614),
 'VRTG_std': np.float64(-0.037226501508413334),
 'FLAP_mean': np.float64(-0.035909836120497474),
 'TRK_std': np.float64(0.03333559841745487),
 'VRTG_mean': np.float64(-0.03199037970377651),
 'WD_std': np.float64(0.019952000406363777),
 'PTCH_mean': np.float64(0.017416942412486425),
 'ROLL_mean': np.float64(0.010664611750704302)}

__Statistical Significance__

In [ ]:
statSignDictLINGAM = {k:result['TouchdownDistance']['pvalue_dict'][k] for k in result['TouchdownDistance']['parents']}
statSignDictLINGAM

{'IVV_mean': np.float64(1.9813955722142226e-118),
 'GS_std': np.float64(1.208852974147172e-189),
 'WS_mean': np.float64(1.007432632527756e-154),
 'PTCH_std': np.float64(3.418818686213332e-48),
 'WD_mean': np.float64(2.691296087381637e-107),
 'CAS_mean': np.float64(8.323537542086485e-16),
 'WOW_std': np.float64(2.677320377594341e-16),
 'WS_std': np.float64(6.860467086401912e-53),
 'WOW_mean': np.float64(2.686619940369558e-09),
 'TAT_std': np.float64(4.7150544216630475e-12),
 'SAT_std': np.float64(2.251140741908461e-09),
 'TH_std': np.float64(1.7952643774717586e-22),
 'VRTG_std': np.float64(0.002367688086373581),
 'FLAP_mean': np.float64(3.616444436816775e-12),
 'TRK_std': np.float64(1.386983691069475e-12),
 'VRTG_mean': np.float64(0.011255873146729576),
 'WD_std': np.float64(6.134180559594017e-06),
 'PTCH_mean': np.float64(0.0015267883275583139),
 'ROLL_mean': np.float64(0.0253398323529417)}

In [ ]:
# Sort by ascending p-value
sorted_pvalues = dict(sorted(statSignDictLINGAM.items(), key=lambda x: x[1]))

# Display
sorted_pvalues

{'GS_std': np.float64(1.208852974147172e-189),
 'WS_mean': np.float64(1.007432632527756e-154),
 'IVV_mean': np.float64(1.9813955722142226e-118),
 'WD_mean': np.float64(2.691296087381637e-107),
 'WS_std': np.float64(6.860467086401912e-53),
 'PTCH_std': np.float64(3.418818686213332e-48),
 'TH_std': np.float64(1.7952643774717586e-22),
 'WOW_std': np.float64(2.677320377594341e-16),
 'CAS_mean': np.float64(8.323537542086485e-16),
 'TRK_std': np.float64(1.386983691069475e-12),
 'FLAP_mean': np.float64(3.616444436816775e-12),
 'TAT_std': np.float64(4.7150544216630475e-12),
 'SAT_std': np.float64(2.251140741908461e-09),
 'WOW_mean': np.float64(2.686619940369558e-09),
 'WD_std': np.float64(6.134180559594017e-06),
 'PTCH_mean': np.float64(0.0015267883275583139),
 'VRTG_std': np.float64(0.002367688086373581),
 'VRTG_mean': np.float64(0.011255873146729576),
 'ROLL_mean': np.float64(0.0253398323529417)}

In [44]:
documentationDict = {}
with open("variable_documentation.txt", "r") as f:
    for line in f:
        varName, desc = line.split(':')
        documentationDict[varName.strip()] = desc.strip()

In [ ]:
mainVars = set(['_'.join(v.split('_')[:-1]) for v in result['TouchdownDistance']['parents']])
for var in mainVars:
    print(f"{var}: {documentationDict[var]}")

In [46]:
for param in givenParams:
    print(f"{param} : {documentationDict[param]}")

WOW : WEIGHT ON WHEELS
LATP : LATITUDE POSITION LSP
LONP : LONGITUDE POSITION LSP
BAL1 : BARO CORRECT ALTITUDE LSP
ROLL : ROLL ANGLE LSP
PTCH : PITCH ANGLE LSP
IVV : INERTIAL VERTICAL SPEED LSP
GS : GROUND SPEED LSP
CAS : COMPUTED AIRSPEED LSP
VRTG : VERTICAL ACCELERATION
LATG : LATERAL ACCELERATION
FLAP : T.E. FLAP POSITION
PLA_2 : POWER LEVER ANGLE 2
PLA_3 : POWER LEVER ANGLE 3
PLA_4 : POWER LEVER ANGLE 4
TRK : TRACK ANGLE TRUE LSP
TH : TRUE HEADING LSP
WS : WIND SPEED
WD : WIND DIRECTION TRUE
TAT : TOTAL AIR TEMPERATURE
SAT : STATIC AIR TEMPERATURE
LOC : LOCALIZER DEVIATION
GLS : GLIDESLOPE DEVIATION


### Using the __LINGAM__ Algorithm (removing __outliers__)

In [11]:
from causalai.models.tabular.lingam import LINGAM

from causalai.data.data_generator import DataGenerator
# also importing data object, data transform object, and prior knowledge object, and the graph plotting function
from causalai.data.tabular import TabularData
from causalai.data.transforms.tabular import StandardizeTransform
from causalai.models.common.prior_knowledge import PriorKnowledge
from causalai.misc.misc import plot_graph, get_precision_recall, get_precision_recall_skeleton, make_symmetric

In [12]:
# 1.
StandardizeTransform_ = StandardizeTransform()
StandardizeTransform_.fit(df.to_numpy())

data_trans = StandardizeTransform_.transform(df.to_numpy())

# 2.
data_obj = TabularData(data_trans, var_names=var_names)

In [13]:
prior_knowledge = None

lingam = LINGAM(
        data=data_obj,
        prior_knowledge=prior_knowledge,
        )
tic = time.time()
result = lingam.run(pvalue_thres=0.05)
toc = time.time()
print(f'Time taken: {toc-tic:.2f}s\n')


print(f' The output result has keys: {result.keys()}')
print(f' The output result["WOW_mean"] has keys: {result["WOW_mean"].keys()}')

Time taken: 254.82s

 The output result has keys: dict_keys(['WOW_mean', 'LATP_mean', 'LONP_mean', 'BAL1_mean', 'ROLL_mean', 'PTCH_mean', 'IVV_mean', 'GS_mean', 'CAS_mean', 'VRTG_mean', 'LATG_mean', 'FLAP_mean', 'PLA_2_mean', 'PLA_3_mean', 'PLA_4_mean', 'TRK_mean', 'TH_mean', 'WS_mean', 'WD_mean', 'TAT_mean', 'SAT_mean', 'LOC_mean', 'GLS_mean', 'WOW_std', 'LATP_std', 'LONP_std', 'BAL1_std', 'ROLL_std', 'PTCH_std', 'IVV_std', 'GS_std', 'CAS_std', 'VRTG_std', 'LATG_std', 'FLAP_std', 'PLA_2_std', 'PLA_3_std', 'PLA_4_std', 'TRK_std', 'TH_std', 'WS_std', 'WD_std', 'TAT_std', 'SAT_std', 'LOC_std', 'GLS_std', 'TouchdownDistance'])
 The output result["WOW_mean"] has keys: dict_keys(['value_dict', 'pvalue_dict', 'parents'])


In [21]:
print(f'Predicted parents:')
result['TouchdownDistance']['parents']

Predicted parents:


['IVV_mean',
 'WS_mean',
 'CAS_mean',
 'WD_mean',
 'PTCH_std',
 'PTCH_mean',
 'WS_std',
 'FLAP_mean',
 'TRK_std',
 'ROLL_mean',
 'TH_std',
 'WD_std',
 'FLAP_std']

__Causal Strengths__

In [22]:
causalStrengthsDictLINGAM = {k:result['TouchdownDistance']['value_dict'][k] for k in result['TouchdownDistance']['parents']}
causalStrengthsDictLINGAM

{'IVV_mean': np.float64(0.5335253913660237),
 'WS_mean': np.float64(-0.2546709911704416),
 'CAS_mean': np.float64(0.139668045631192),
 'WD_mean': np.float64(0.11392126844994209),
 'PTCH_std': np.float64(0.07701121119796996),
 'PTCH_mean': np.float64(-0.06439344956861587),
 'WS_std': np.float64(0.05008136863988226),
 'FLAP_mean': np.float64(-0.03232475549604351),
 'TRK_std': np.float64(0.028462330542914978),
 'ROLL_mean': np.float64(0.027736357281520668),
 'TH_std': np.float64(0.025989169103310186),
 'WD_std': np.float64(0.015915464915282417),
 'FLAP_std': np.float64(0.010571395265485017)}

In [23]:
# Sorting the dictionary by absolute value in descending order
sorted_dict = dict(sorted(causalStrengthsDictLINGAM.items(), key=lambda item: abs(item[1]), reverse=True))

# Printing the sorted dictionary
sorted_dict

{'IVV_mean': np.float64(0.5335253913660237),
 'WS_mean': np.float64(-0.2546709911704416),
 'CAS_mean': np.float64(0.139668045631192),
 'WD_mean': np.float64(0.11392126844994209),
 'PTCH_std': np.float64(0.07701121119796996),
 'PTCH_mean': np.float64(-0.06439344956861587),
 'WS_std': np.float64(0.05008136863988226),
 'FLAP_mean': np.float64(-0.03232475549604351),
 'TRK_std': np.float64(0.028462330542914978),
 'ROLL_mean': np.float64(0.027736357281520668),
 'TH_std': np.float64(0.025989169103310186),
 'WD_std': np.float64(0.015915464915282417),
 'FLAP_std': np.float64(0.010571395265485017)}

__Statistical Significance__

In [24]:
statSignDictLINGAM = {k:result['TouchdownDistance']['pvalue_dict'][k] for k in result['TouchdownDistance']['parents']}
statSignDictLINGAM

{'IVV_mean': np.float64(0.0),
 'WS_mean': np.float64(4.576835315981464e-256),
 'CAS_mean': np.float64(4.092288958822579e-34),
 'WD_mean': np.float64(9.36364344133818e-134),
 'PTCH_std': np.float64(9.488045643802794e-26),
 'PTCH_mean': np.float64(4.015401097088567e-33),
 'WS_std': np.float64(1.4817880496144612e-18),
 'FLAP_mean': np.float64(5.070544770728853e-11),
 'TRK_std': np.float64(4.1182737068990113e-10),
 'ROLL_mean': np.float64(2.4417829734712072e-09),
 'TH_std': np.float64(1.1377763977212215e-07),
 'WD_std': np.float64(0.0002192950129951844),
 'FLAP_std': np.float64(0.03204168479238425)}

In [25]:
# Sort by ascending p-value
sorted_pvalues = dict(sorted(statSignDictLINGAM.items(), key=lambda x: x[1]))

# Display
sorted_pvalues

{'IVV_mean': np.float64(0.0),
 'WS_mean': np.float64(4.576835315981464e-256),
 'WD_mean': np.float64(9.36364344133818e-134),
 'CAS_mean': np.float64(4.092288958822579e-34),
 'PTCH_mean': np.float64(4.015401097088567e-33),
 'PTCH_std': np.float64(9.488045643802794e-26),
 'WS_std': np.float64(1.4817880496144612e-18),
 'FLAP_mean': np.float64(5.070544770728853e-11),
 'TRK_std': np.float64(4.1182737068990113e-10),
 'ROLL_mean': np.float64(2.4417829734712072e-09),
 'TH_std': np.float64(1.1377763977212215e-07),
 'WD_std': np.float64(0.0002192950129951844),
 'FLAP_std': np.float64(0.03204168479238425)}

In [26]:
documentationDict = {}
with open("variable_documentation.txt", "r") as f:
    for line in f:
        varName, desc = line.split(':')
        documentationDict[varName.strip()] = desc.strip()

In [27]:
mainVars = set(['_'.join(v.split('_')[:-1]) for v in result['TouchdownDistance']['parents']])
for var in mainVars:
    print(f"{var}: {documentationDict[var]}")

ROLL: ROLL ANGLE LSP
FLAP: T.E. FLAP POSITION
WS: WIND SPEED
CAS: COMPUTED AIRSPEED LSP
PTCH: PITCH ANGLE LSP
TRK: TRACK ANGLE TRUE LSP
WD: WIND DIRECTION TRUE
IVV: INERTIAL VERTICAL SPEED LSP
TH: TRUE HEADING LSP


### Using the __PC__ Algorithm

In [27]:
from causalai.models.tabular.pc import PCSingle, PC
from causalai.models.common.CI_tests.partial_correlation import PartialCorrelation
from causalai.models.common.CI_tests.discrete_ci_tests import DiscreteCI_tests
from causalai.models.common.CI_tests.kci import KCI


# also importing data object, data transform object, and prior knowledge object, and the graph plotting function
from causalai.data.data_generator import DataGenerator, GenerateRandomTabularSEM
from causalai.data.tabular import TabularData
from causalai.data.transforms.time_series import StandardizeTransform
from causalai.models.common.prior_knowledge import PriorKnowledge
from causalai.misc.misc import plot_graph, get_precision_recall, get_precision_recall_skeleton, make_symmetric

In [28]:
# 1.
StandardizeTransform_ = StandardizeTransform()
StandardizeTransform_.fit(df.to_numpy())

data_trans = StandardizeTransform_.transform(df.to_numpy())

# 2.
data_obj = TabularData(data_trans, var_names=var_names)

In [29]:
prior_knowledge = None #  PriorKnowledge(forbidden_links={'a': ['b']})

pvalue_thres = 0.05
CI_test = PartialCorrelation()
# CI_test = KCI(chunk_size=100) # use if the causal relationship is expected to be non-linear
pc = PC(
        data=data_obj,
        prior_knowledge=prior_knowledge,
        CI_test=CI_test,
        use_multiprocessing=True
        )

In [30]:
tic = time.time()


result = pc.run(pvalue_thres=pvalue_thres, max_condition_set_size=2)

toc = time.time()

print(f'Time taken: {toc-tic:.2f}s\n')

print(f' The output result has keys: {result.keys()}')
print(f' The output result["WOW_mean"] has keys: {result["WOW_mean"].keys()}')

2025-04-14 15:59:29,185	INFO worker.py:1852 -- Started a local Ray instance.


Time taken: 543.24s

 The output result has keys: dict_keys(['WOW_mean', 'LATP_mean', 'LONP_mean', 'BAL1_mean', 'ROLL_mean', 'PTCH_mean', 'IVV_mean', 'GS_mean', 'CAS_mean', 'VRTG_mean', 'LATG_mean', 'FLAP_mean', 'PLA_2_mean', 'PLA_3_mean', 'PLA_4_mean', 'TRK_mean', 'TH_mean', 'WS_mean', 'WD_mean', 'TAT_mean', 'SAT_mean', 'LOC_mean', 'GLS_mean', 'WOW_std', 'LATP_std', 'LONP_std', 'BAL1_std', 'ROLL_std', 'PTCH_std', 'IVV_std', 'GS_std', 'CAS_std', 'VRTG_std', 'LATG_std', 'FLAP_std', 'PLA_2_std', 'PLA_3_std', 'PLA_4_std', 'TRK_std', 'TH_std', 'WS_std', 'WD_std', 'TAT_std', 'SAT_std', 'LOC_std', 'GLS_std', 'TouchdownDistance'])
 The output result["WOW_mean"] has keys: dict_keys(['parents', 'value_dict', 'pvalue_dict'])


In [34]:
result['TouchdownDistance']['parents']

['GS_mean',
 'WD_std',
 'BAL1_std',
 'TH_std',
 'LATP_mean',
 'FLAP_mean',
 'TH_mean',
 'TRK_std',
 'GLS_mean',
 'GS_std',
 'CAS_std',
 'PTCH_mean',
 'WOW_std',
 'WD_mean']

__Causal Strengths__

In [31]:
# Sorting the dictionary by absolute value in descending order
sorted_dict = dict(sorted(result['TouchdownDistance']['value_dict'].items(), key=lambda item: abs(item[1]), reverse=True))

# Printing the sorted dictionary
sorted_dict

{'BAL1_std': np.float64(-0.12688534595250892),
 'GLS_mean': np.float64(0.11266157453989827),
 'GS_std': np.float64(0.11018961483378747),
 'GS_mean': np.float64(0.0778143783074426),
 'TH_mean': np.float64(0.06493571268276183),
 'LATP_mean': np.float64(-0.06028400057041693),
 'WD_std': np.float64(0.039922744860291896),
 'WD_mean': np.float64(0.03369230117074389),
 'WOW_std': np.float64(0.031528762867769206),
 'TH_std': np.float64(0.02666236113831115),
 'CAS_std': np.float64(0.022783764216865645),
 'FLAP_mean': np.float64(-0.02120427101248704),
 'TRK_std': np.float64(0.012959853525848282)}

__Statistical Significance__

In [32]:
# Assuming this is your pvalue_dict:
pvalue_dict = result['TouchdownDistance']['pvalue_dict']

# Sort by ascending p-value
sorted_pvalues = dict(sorted(pvalue_dict.items(), key=lambda x: x[1]))

# Display
sorted_pvalues

{'BAL1_std': np.float64(7.257167436596981e-160),
 'GLS_mean': np.float64(3.0347749363048514e-126),
 'GS_std': np.float64(8.09763130617221e-121),
 'GS_mean': np.float64(5.091373341725236e-61),
 'TH_mean': np.float64(5.348706411634856e-43),
 'LATP_mean': np.float64(2.752187408344517e-37),
 'WD_std': np.float64(3.021525017500682e-17),
 'WD_mean': np.float64(1.0269216725912648e-12),
 'WOW_std': np.float64(2.576115264754645e-11),
 'TH_std': np.float64(1.708171709126981e-08),
 'CAS_std': np.float64(1.4453218768032563e-06),
 'FLAP_mean': np.float64(7.307888495063369e-06),
 'TRK_std': np.float64(0.006130793387893033)}

In [ ]:
documentationDict = {}
with open("variable_documentation.txt", "r") as f:
    for line in f:
        varName, desc = line.split(':')
        documentationDict[varName.strip()] = desc.strip()

In [ ]:
mainVars = set(['_'.join(v.split('_')[:-1]) for v in result['TouchdownDistance']['parents']])
for var in mainVars:
    print(f"{var}: {documentationDict[var]}")

CAS: COMPUTED AIRSPEED LSP
WOW: WEIGHT ON WHEELS
TRK: TRACK ANGLE TRUE LSP
PTCH: PITCH ANGLE LSP
WD: WIND DIRECTION TRUE
FLAP: T.E. FLAP POSITION
GS: GROUND SPEED LSP
TH: TRUE HEADING LSP
LATP: LATITUDE POSITION LSP
GLS: GLIDESLOPE DEVIATION
BAL1: BARO CORRECT ALTITUDE LSP


### Using the __PC__ Algorithm (removing __outliers__)

In [4]:
from causalai.models.tabular.pc import PCSingle, PC
from causalai.models.common.CI_tests.partial_correlation import PartialCorrelation
from causalai.models.common.CI_tests.discrete_ci_tests import DiscreteCI_tests
from causalai.models.common.CI_tests.kci import KCI


# also importing data object, data transform object, and prior knowledge object, and the graph plotting function
from causalai.data.data_generator import DataGenerator, GenerateRandomTabularSEM
from causalai.data.tabular import TabularData
from causalai.data.transforms.time_series import StandardizeTransform
from causalai.models.common.prior_knowledge import PriorKnowledge
from causalai.misc.misc import plot_graph, get_precision_recall, get_precision_recall_skeleton, make_symmetric

In [15]:
from causallearn.search.ConstraintBased.PC import pc
from causallearn.utils.GraphUtils import GraphUtils
from causallearn.utils.cit import fisherz

In [5]:
# 1.
StandardizeTransform_ = StandardizeTransform()
StandardizeTransform_.fit(df.to_numpy())

data_trans = StandardizeTransform_.transform(df.to_numpy())

# 2.
data_obj = TabularData(data_trans, var_names=var_names)

In [9]:
prior_knowledge = None #  PriorKnowledge(forbidden_links={'a': ['b']})

pvalue_thres = 0.05
CI_test = PartialCorrelation()
# CI_test = KCI(chunk_size=100) # use if the causal relationship is expected to be non-linear
pc = PC(
        data=data_obj,
        prior_knowledge=prior_knowledge,
        CI_test=CI_test,
        use_multiprocessing=True
        )

In [10]:
tic = time.time()


result = pc.run(pvalue_thres=pvalue_thres, max_condition_set_size=2)

toc = time.time()

print(f'Time taken: {toc-tic:.2f}s\n')

print(f' The output result has keys: {result.keys()}')
print(f' The output result["WOW_mean"] has keys: {result["WOW_mean"].keys()}')

2025-04-30 13:47:03,399	INFO worker.py:1852 -- Started a local Ray instance.


Time taken: 902.02s

 The output result has keys: dict_keys(['WOW_mean', 'LATP_mean', 'LONP_mean', 'BAL1_mean', 'ROLL_mean', 'PTCH_mean', 'IVV_mean', 'GS_mean', 'CAS_mean', 'VRTG_mean', 'LATG_mean', 'FLAP_mean', 'PLA_2_mean', 'PLA_3_mean', 'PLA_4_mean', 'TRK_mean', 'TH_mean', 'WS_mean', 'WD_mean', 'TAT_mean', 'SAT_mean', 'LOC_mean', 'GLS_mean', 'WOW_std', 'LATP_std', 'LONP_std', 'BAL1_std', 'ROLL_std', 'PTCH_std', 'IVV_std', 'GS_std', 'CAS_std', 'VRTG_std', 'LATG_std', 'FLAP_std', 'PLA_2_std', 'PLA_3_std', 'PLA_4_std', 'TRK_std', 'TH_std', 'WS_std', 'WD_std', 'TAT_std', 'SAT_std', 'LOC_std', 'GLS_std', 'TouchdownDistance'])
 The output result["WOW_mean"] has keys: dict_keys(['parents', 'value_dict', 'pvalue_dict'])


In [16]:
GraphUtils.draw_graph(result.G)

AttributeError: type object 'GraphUtils' has no attribute 'draw_graph'

In [32]:
result['TouchdownDistance']['parents']

['TH_mean',
 'PTCH_mean',
 'GS_mean',
 'GLS_mean',
 'WD_mean',
 'GS_std',
 'TRK_std',
 'WD_std',
 'CAS_std',
 'ROLL_mean',
 'BAL1_std',
 'TH_std',
 'FLAP_mean',
 'TRK_mean',
 'WS_mean']

__Causal Strengths__

In [33]:
# Sorting the dictionary by absolute value in descending order
sorted_dict = dict(sorted(result['TouchdownDistance']['value_dict'].items(), key=lambda item: abs(item[1]), reverse=True))

# Printing the sorted dictionary
sorted_dict

{'GLS_mean': np.float64(0.13673530016721316),
 'BAL1_std': np.float64(-0.13465520369070255),
 'GS_std': np.float64(0.12626692582328836),
 'GS_mean': np.float64(0.08091930137980746),
 'WD_std': np.float64(0.040635401796549786),
 'CAS_std': np.float64(0.03910470068757439),
 'WS_mean': np.float64(-0.03385979546962218),
 'WD_mean': np.float64(0.03331150372671772),
 'ROLL_mean': np.float64(0.02369618811082912),
 'TH_mean': np.float64(0.017379085363308382),
 'TH_std': np.float64(0.015703504952765294),
 'FLAP_mean': np.float64(-0.012992223073320236),
 'TRK_mean': np.float64(0.01197580579213217),
 'TRK_std': np.float64(0.009328644226344872)}

__Statistical Significance__

In [34]:
# Assuming this is your pvalue_dict:
pvalue_dict = result['TouchdownDistance']['pvalue_dict']

# Sort by ascending p-value
sorted_pvalues = dict(sorted(pvalue_dict.items(), key=lambda x: x[1]))

# Display
sorted_pvalues

{'GLS_mean': np.float64(2.1343457945785342e-190),
 'BAL1_std': np.float64(1.1689567317648393e-184),
 'GS_std': np.float64(1.9390659868738262e-162),
 'GS_mean': np.float64(1.535238718378641e-67),
 'WD_std': np.float64(3.0601531003766313e-18),
 'CAS_std': np.float64(5.253349431483238e-17),
 'WS_mean': np.float64(3.9840447605764175e-13),
 'WD_mean': np.float64(9.432865171794687e-13),
 'ROLL_mean': np.float64(3.8280886806401636e-07),
 'TH_mean': np.float64(0.00019646070028771687),
 'TH_std': np.float64(0.000766881774241568),
 'FLAP_mean': np.float64(0.005377045827869719),
 'TRK_mean': np.float64(0.010295213333268511),
 'TRK_std': np.float64(0.04565333902686983)}

In [35]:
documentationDict = {}
with open("variable_documentation.txt", "r") as f:
    for line in f:
        varName, desc = line.split(':')
        documentationDict[varName.strip()] = desc.strip()

In [36]:
mainVars = set(['_'.join(v.split('_')[:-1]) for v in result['TouchdownDistance']['parents']])
for var in mainVars:
    print(f"{var}: {documentationDict[var]}")

ROLL: ROLL ANGLE LSP
FLAP: T.E. FLAP POSITION
WS: WIND SPEED
CAS: COMPUTED AIRSPEED LSP
GLS: GLIDESLOPE DEVIATION
PTCH: PITCH ANGLE LSP
TRK: TRACK ANGLE TRUE LSP
WD: WIND DIRECTION TRUE
GS: GROUND SPEED LSP
BAL1: BARO CORRECT ALTITUDE LSP
TH: TRUE HEADING LSP


### Using the __Grow-Shrink__ Algorithm

In [5]:
from causalai.models.tabular.grow_shrink import GrowShrink
from causalai.models.common.CI_tests.partial_correlation import PartialCorrelation
from causalai.models.common.CI_tests.discrete_ci_tests import DiscreteCI_tests
from causalai.models.common.CI_tests.kci import KCI


# also importing data object, and prior knowledge object, and the graph plotting function
from causalai.data.data_generator import DataGenerator, GenerateRandomTabularSEM
from causalai.data.tabular import TabularData
from causalai.data.transforms.time_series import StandardizeTransform
from causalai.models.common.prior_knowledge import PriorKnowledge
from causalai.misc.misc import plot_graph, get_precision_recall, get_precision_recall_skeleton, make_symmetric,\
                               _get_precision_recall_single

# Helper functions to compute ground truth

from causalai.tests.data.transforms.networkx_helper_functions import causalai2networkx, compute_markov_blanket

In [6]:
# 1.
StandardizeTransform_ = StandardizeTransform()
StandardizeTransform_.fit(df.to_numpy())

data_trans = StandardizeTransform_.transform(df.to_numpy())

# 2.
data_obj = TabularData(data_trans, var_names=var_names)

In [7]:
prior_knowledge = None #  PriorKnowledge(forbidden_links={'a': ['b']})

pvalue_thres = 0.05
CI_test = PartialCorrelation()
# CI_test = KCI(chunk_size=100) # use if the causal relationship is expected to be non-linear
gs = GrowShrink(
        data=data_obj,
        prior_knowledge=prior_knowledge,
        CI_test=CI_test,
        use_multiprocessing=True,
        update_shrink=False
        )

target_var = 'TouchdownDistance'

In [8]:
tic = time.time()


result = gs.run(target_var=target_var, pvalue_thres=pvalue_thres)

toc = time.time()

print(f'Time taken: {toc-tic:.2f}s\n')

print(f" target var {target_var}\'s estimated MB is: {result['markov_blanket']}")

2025-04-14 15:17:03,816	INFO worker.py:1852 -- Started a local Ray instance.


Time taken: 11.94s

 target var TouchdownDistance's estimated MB is: ['IVV_mean', 'WOW_mean', 'TRK_mean', 'LATP_mean', 'TRK_std', 'WOW_std', 'FLAP_std', 'ROLL_mean', 'LATP_std', 'BAL1_mean', 'VRTG_std', 'PLA_3_mean', 'LONP_mean', 'GS_std', 'IVV_std', 'BAL1_std', 'TAT_std', 'GS_mean', 'CAS_mean', 'LONP_std', 'TH_mean', 'WS_mean', 'WD_mean', 'PLA_2_mean', 'WD_std', 'ROLL_std', 'PLA_2_std', 'GLS_mean', 'SAT_mean', 'WS_std', 'CAS_std', 'TH_std', 'GLS_std', 'SAT_std', 'PTCH_std', 'TAT_mean']


In [11]:
result['markov_blanket']

['IVV_mean',
 'WOW_mean',
 'TRK_mean',
 'LATP_mean',
 'TRK_std',
 'WOW_std',
 'FLAP_std',
 'ROLL_mean',
 'LATP_std',
 'BAL1_mean',
 'VRTG_std',
 'PLA_3_mean',
 'LONP_mean',
 'GS_std',
 'IVV_std',
 'BAL1_std',
 'TAT_std',
 'GS_mean',
 'CAS_mean',
 'LONP_std',
 'TH_mean',
 'WS_mean',
 'WD_mean',
 'PLA_2_mean',
 'WD_std',
 'ROLL_std',
 'PLA_2_std',
 'GLS_mean',
 'SAT_mean',
 'WS_std',
 'CAS_std',
 'TH_std',
 'GLS_std',
 'SAT_std',
 'PTCH_std',
 'TAT_mean']

__Causal Strengths__

In [21]:
# Extract variable names and their causal strengths
var_strength_pairs = [(var, result['value_dict'][var]) for var in result['markov_blanket']]

# Sort based on absolute value of causal strength, descending
var_strength_pairs_sorted = sorted(var_strength_pairs, key=lambda x: abs(x[1]), reverse=True)

# Print the sorted results
for var, strength in var_strength_pairs_sorted:
    print(f"{var}: {strength:.4f}")

BAL1_std: -0.1768
GS_mean: 0.1702
GLS_mean: 0.1025
GS_std: 0.0837
WOW_mean: -0.0806
WS_std: 0.0768
TH_mean: 0.0751
LATP_std: -0.0733
BAL1_mean: -0.0724
WOW_std: -0.0593
LONP_std: -0.0509
CAS_mean: -0.0482
TH_std: 0.0468
WD_mean: 0.0423
SAT_mean: -0.0422
TAT_mean: 0.0418
TAT_std: 0.0372
PTCH_std: -0.0370
FLAP_std: 0.0351
TRK_std: 0.0335
SAT_std: -0.0330
LATP_mean: -0.0271
IVV_mean: -0.0268
WD_std: 0.0265
ROLL_std: -0.0211
GLS_std: 0.0210
LONP_mean: -0.0205
ROLL_mean: 0.0192
PLA_2_std: 0.0187
WS_mean: -0.0182
IVV_std: -0.0154
CAS_std: -0.0149
TRK_mean: -0.0140
VRTG_std: -0.0124
PLA_3_mean: -0.0114
PLA_2_mean: 0.0096


__Statistical Significance__

In [23]:
# Extract variable names and their p-values
var_pval_pairs = [(var, result['pvalue_dict'][var]) for var in result['markov_blanket']]

# Sort based on p-value in ascending order
var_pval_pairs_sorted = sorted(var_pval_pairs, key=lambda x: x[1])

# Print the sorted results
for var, pval in var_pval_pairs_sorted:
    print(f"{var}: {pval}")

BAL1_std: 0.0
GS_mean: 9.618290328644043e-288
GLS_mean: 1.387904526042918e-104
GS_std: 2.5705429012139518e-70
WOW_mean: 2.8273272120400033e-65
WS_std: 1.6892370311021886e-59
TH_mean: 6.182064272879291e-57
LATP_std: 3.060685568340481e-54
BAL1_mean: 5.701202633502635e-53
WOW_std: 3.848802736831354e-36
LONP_std: 5.053317397507899e-27
CAS_mean: 2.2689575884273127e-24
TH_std: 4.568469472383052e-23
WD_mean: 3.9276054479445213e-19
SAT_mean: 4.588429764990772e-19
TAT_mean: 8.91310138663588e-19
TAT_std: 3.4136746149814997e-15
PTCH_std: 5.228187917003706e-15
FLAP_std: 1.0817016786025936e-13
TRK_std: 1.3711414790597732e-12
SAT_std: 2.857042618895004e-12
LATP_mean: 1.062062241396898e-08
IVV_mean: 1.4201119350044628e-08
WD_std: 2.2045520123744288e-08
ROLL_std: 8.376579278929784e-06
GLS_std: 9.339685311034193e-06
LONP_mean: 1.5290577347067948e-05
ROLL_mean: 5.14347327665195e-05
PLA_2_std: 7.859458795370494e-05
WS_mean: 0.00011535698016567609
IVV_std: 0.0010920176880496505
CAS_std: 0.0016612072940982

### Using the __Grow-Shrink__ Algorithm (removing __outliers__)

In [37]:
from causalai.models.tabular.grow_shrink import GrowShrink
from causalai.models.common.CI_tests.partial_correlation import PartialCorrelation
from causalai.models.common.CI_tests.discrete_ci_tests import DiscreteCI_tests
from causalai.models.common.CI_tests.kci import KCI


# also importing data object, and prior knowledge object, and the graph plotting function
from causalai.data.data_generator import DataGenerator, GenerateRandomTabularSEM
from causalai.data.tabular import TabularData
from causalai.data.transforms.time_series import StandardizeTransform
from causalai.models.common.prior_knowledge import PriorKnowledge
from causalai.misc.misc import plot_graph, get_precision_recall, get_precision_recall_skeleton, make_symmetric,\
                               _get_precision_recall_single

# Helper functions to compute ground truth

from causalai.tests.data.transforms.networkx_helper_functions import causalai2networkx, compute_markov_blanket

In [38]:
# 1.
StandardizeTransform_ = StandardizeTransform()
StandardizeTransform_.fit(df.to_numpy())

data_trans = StandardizeTransform_.transform(df.to_numpy())

# 2.
data_obj = TabularData(data_trans, var_names=var_names)

In [39]:
prior_knowledge = None #  PriorKnowledge(forbidden_links={'a': ['b']})

pvalue_thres = 0.05
CI_test = PartialCorrelation()
# CI_test = KCI(chunk_size=100) # use if the causal relationship is expected to be non-linear
gs = GrowShrink(
        data=data_obj,
        prior_knowledge=prior_knowledge,
        CI_test=CI_test,
        use_multiprocessing=True,
        update_shrink=False
        )

target_var = 'TouchdownDistance'

In [40]:
tic = time.time()


result = gs.run(target_var=target_var, pvalue_thres=pvalue_thres)

toc = time.time()

print(f'Time taken: {toc-tic:.2f}s\n')

print(f" target var {target_var}\'s estimated MB is: {result['markov_blanket']}")

2025-04-21 17:49:26,109	INFO worker.py:1852 -- Started a local Ray instance.


Time taken: 13.55s

 target var TouchdownDistance's estimated MB is: ['TAT_std', 'TH_mean', 'ROLL_std', 'TAT_mean', 'WOW_std', 'BAL1_mean', 'TRK_mean', 'BAL1_std', 'SAT_std', 'IVV_std', 'WS_std', 'TH_std', 'LATP_mean', 'PTCH_std', 'GS_mean', 'LONP_mean', 'CAS_mean', 'TRK_std', 'PLA_3_mean', 'ROLL_mean', 'LATP_std', 'WD_mean', 'PLA_2_std', 'WS_mean', 'LOC_std', 'WOW_mean', 'LONP_std', 'GLS_mean', 'SAT_mean', 'GS_std', 'FLAP_std', 'WD_std', 'GLS_std']


In [41]:
result['markov_blanket']

['TAT_std',
 'TH_mean',
 'ROLL_std',
 'TAT_mean',
 'WOW_std',
 'BAL1_mean',
 'TRK_mean',
 'BAL1_std',
 'SAT_std',
 'IVV_std',
 'WS_std',
 'TH_std',
 'LATP_mean',
 'PTCH_std',
 'GS_mean',
 'LONP_mean',
 'CAS_mean',
 'TRK_std',
 'PLA_3_mean',
 'ROLL_mean',
 'LATP_std',
 'WD_mean',
 'PLA_2_std',
 'WS_mean',
 'LOC_std',
 'WOW_mean',
 'LONP_std',
 'GLS_mean',
 'SAT_mean',
 'GS_std',
 'FLAP_std',
 'WD_std',
 'GLS_std']

__Causal Strengths__

In [42]:
# Extract variable names and their causal strengths
var_strength_pairs = [(var, result['value_dict'][var]) for var in result['markov_blanket']]

# Sort based on absolute value of causal strength, descending
var_strength_pairs_sorted = sorted(var_strength_pairs, key=lambda x: abs(x[1]), reverse=True)

# Print the sorted results
for var, strength in var_strength_pairs_sorted:
    print(f"{var}: {strength:.4f}")

BAL1_std: -0.2058
GS_mean: 0.2030
GLS_mean: 0.1412
WOW_std: -0.1401
WOW_mean: -0.1108
BAL1_mean: -0.1033
GS_std: 0.0925
LONP_std: -0.0839
WS_std: 0.0761
LATP_mean: 0.0703
LATP_std: -0.0660
CAS_mean: -0.0618
TH_mean: 0.0576
WD_mean: 0.0540
PTCH_std: -0.0533
SAT_mean: -0.0453
TAT_mean: 0.0448
FLAP_std: 0.0417
ROLL_mean: 0.0395
WD_std: 0.0354
TRK_std: 0.0336
TAT_std: 0.0313
TH_std: 0.0303
ROLL_std: -0.0295
SAT_std: -0.0286
TRK_mean: 0.0283
PLA_2_std: 0.0263
IVV_std: 0.0254
LONP_mean: -0.0239
PLA_3_mean: -0.0223
LOC_std: 0.0174
GLS_std: 0.0164
WS_mean: -0.0139


__Statistical Significance__

In [43]:
# Extract variable names and their p-values
var_pval_pairs = [(var, result['pvalue_dict'][var]) for var in result['markov_blanket']]

# Sort based on p-value in ascending order
var_pval_pairs_sorted = sorted(var_pval_pairs, key=lambda x: x[1])

# Print the sorted results
for var, pval in var_pval_pairs_sorted:
    print(f"{var}: {pval}")

BAL1_std: 0.0
GS_mean: 0.0
GLS_mean: 5.508960530839164e-203
WOW_std: 1.2140502939442525e-199
WOW_mean: 2.7462371113599695e-125
BAL1_mean: 6.441646806851545e-109
GS_std: 1.1959026822219545e-87
LONP_std: 2.1384792785868653e-72
WS_std: 6.275484447534726e-60
LATP_mean: 2.2994810028287277e-51
LATP_std: 1.6888565923236976e-45
CAS_mean: 4.619060005721376e-40
TH_mean: 5.437414398923863e-35
WD_mean: 5.274134308355578e-31
PTCH_std: 3.432237573700519e-30
SAT_mean: 2.6276347165705715e-22
TAT_mean: 8.646010200554978e-22
FLAP_std: 4.2239401023674854e-19
ROLL_mean: 2.4850262047109708e-17
WD_std: 3.617335939791675e-14
TRK_std: 5.877043038370612e-13
TAT_std: 1.9823618025411302e-11
TH_std: 8.002717666590241e-11
ROLL_std: 2.746113648544801e-10
SAT_std: 9.537444922634637e-10
TRK_mean: 1.3586674420619733e-09
PLA_2_std: 1.7724608393635642e-08
IVV_std: 5.483609030163795e-08
LONP_mean: 2.93995622591639e-07
PLA_3_mean: 1.7954497564819195e-06
LOC_std: 0.00019658262024674095
GLS_std: 0.00044872044425237756
WS_me

## Understanding Conditional Independence and p-values in Causal Discovery

### Scenario:
Suppose we have three variables: **A**, **B**, and **C**. We're trying to understand the causal relationship between **A** and **C**, while considering **B**.

---

### The Null Hypothesis (H₀):
> **A and C are conditionally independent given B**

This means:
> _“Once we know B, knowing A gives us no additional information about C.”_

---

### What does the p-value tell us?

The **p-value** helps assess how compatible our data is with the null hypothesis (H₀).  
In the context of causal discovery, it helps us decide whether the observed relationship between two variables could just be due to chance under the assumption of conditional independence.

---

### Interpreting p-values:

| p-value | Interpretation |
|--------|----------------|
| **Low (e.g., < 0.05)** | Strong evidence **against** H₀ → A and C are **not** conditionally independent given B → suggests a **potential causal relationship** |
| **High (e.g., > 0.05)** | Weak evidence against H₀ → A and C **may be** conditionally independent given B → B might **explain** the relationship between A and C |

---

### Practical Implication:

If the causal structure is:

A-->B-->C

Then A and C are **dependent** in general.  
But **once we condition on B**, A and C become **independent** — there's **no direct path** from A to C.

This principle helps algorithms like **LiNGAM** discover and remove **indirect or spurious causal links**, keeping only the **true direct causal influences**.

---

### 📌 Summary:
- **p-value** evaluates whether two variables are **conditionally independent** given others.
- A **low p-value** → evidence of a **direct causal effect**.
- A **high p-value** → suggests the effect might be explained by **indirect paths** or **confounders**.